# Day 6 — Data Analysis & Data Cleaning
**Date:** 15-Jul-2026

Today's topics:
- NumPy arrays & Broadcasting
- Pandas DataFrame basics
- Reading CSV / Excel / JSON
- Filtering
- GroupBy
- Missing value handling
- Data preprocessing
- Feature engineering basics

This notebook is self-contained: it generates its own sample dataset (and saves it as CSV, Excel, and JSON) so every section runs top-to-bottom without needing any external files.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)


NumPy version: 2.5.1
Pandas version: 3.0.3


## 2. NumPy Arrays

The basic building block of NumPy is the `ndarray`. Let's create a few and inspect their properties.

In [2]:
# 1D, 2D arrays
arr1 = np.array([10, 20, 30, 40, 50])
arr2 = np.array([[1, 2, 3], [4, 5, 6]])

print("arr1:", arr1, "| shape:", arr1.shape, "| dtype:", arr1.dtype)
print("arr2:\n", arr2, "\nshape:", arr2.shape)

# Handy constructors
zeros = np.zeros((2, 3))
ones = np.ones((3, 2))
arange = np.arange(0, 20, 2)
linspace = np.linspace(0, 1, 5)

print("\nzeros:\n", zeros)
print("ones:\n", ones)
print("arange:", arange)
print("linspace:", linspace)


arr1: [10 20 30 40 50] | shape: (5,) | dtype: int64
arr2:
 [[1 2 3]
 [4 5 6]] 
shape: (2, 3)

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1.]
 [1. 1.]
 [1. 1.]]
arange: [ 0  2  4  6  8 10 12 14 16 18]
linspace: [0.   0.25 0.5  0.75 1.  ]


In [3]:
# Indexing, slicing, and vectorized math (no explicit loops needed)
prices = np.array([100, 150, 200, 250, 300])
discount_pct = 10

discounted_prices = prices - (prices * discount_pct / 100)
print("Original prices:", prices)
print("Discounted prices:", discounted_prices)

# Boolean masking
expensive_mask = prices > 150
print("\nItems priced above 150:", prices[expensive_mask])


Original prices: [100 150 200 250 300]
Discounted prices: [ 90. 135. 180. 225. 270.]

Items priced above 150: [200 250 300]


## 3. Broadcasting

Broadcasting lets NumPy perform arithmetic on arrays of different shapes without writing explicit loops, as long as their shapes are "compatible".

In [4]:
# Example 1: scalar broadcast across an array
temps_celsius = np.array([20, 25, 30, 15, 10])
temps_fahrenheit = temps_celsius * 9/5 + 32   # scalar broadcasts to every element
print("Celsius:", temps_celsius)
print("Fahrenheit:", temps_fahrenheit)

# Example 2: (3,4) matrix + (4,) row vector -> row vector is broadcast to every row
sales = np.array([
    [100, 200, 300, 400],
    [110, 210, 310, 410],
    [120, 220, 320, 420],
])
monthly_bonus = np.array([10, 20, 30, 40])   # shape (4,)

sales_with_bonus = sales + monthly_bonus
print("\nSales:\n", sales)
print("\nSales + bonus (broadcasted):\n", sales_with_bonus)

# Example 3: (3,1) column vector + (4,) row vector -> outer-sum via broadcasting
col = np.array([[1], [2], [3]])   # shape (3,1)
row = np.array([100, 200, 300, 400])  # shape (4,)
print("\nOuter broadcast result shape:", (col + row).shape)
print(col + row)


Celsius: [20 25 30 15 10]
Fahrenheit: [68. 77. 86. 59. 50.]

Sales:
 [[100 200 300 400]
 [110 210 310 410]
 [120 220 320 420]]

Sales + bonus (broadcasted):
 [[110 220 330 440]
 [120 230 340 450]
 [130 240 350 460]]

Outer broadcast result shape: (3, 4)
[[101 201 301 401]
 [102 202 302 402]
 [103 203 303 403]]


## 4. Creating a Sample "Messy" Dataset

We'll simulate a realistic messy dataset (customer orders) with:
- Missing values
- Inconsistent text casing
- Duplicate rows
- Mixed data types

Then we'll save it as CSV, Excel, and JSON so we can practice reading each format.

In [5]:
np.random.seed(42)

n = 20
customer_ids = [f"CUST{100+i}" for i in range(n)]
names = ["Alice", "Bob", "charlie", "DIANA", "Ethan", "fiona", "George", "Hannah",
         "Ivan", "Julia", "Kevin", "Laura", "Mallory", "Nina", "Oscar", "Priya",
         "Quinn", "Rachel", "Sam", "Tina"]

cities = ["Bengaluru", "Mumbai", "Delhi", "Chennai", "Hyderabad", None]
categories = ["Electronics", "Grocery", "Clothing", "Books"]

df_raw = pd.DataFrame({
    "customer_id": customer_ids,
    "name": names,
    "age": np.random.choice([22, 25, 28, np.nan, 35, 40, 45, np.nan, 30, 27], size=n),
    "city": np.random.choice(cities, size=n),
    "category": np.random.choice(categories, size=n),
    "quantity": np.random.choice([1, 2, 3, np.nan, 5], size=n),
    "unit_price": np.round(np.random.uniform(100, 2000, size=n), 2),
    "order_date": pd.date_range("2026-06-01", periods=n, freq="3D").astype(str),
})

# Introduce a duplicate row and a couple of missing prices for realism
df_raw = pd.concat([df_raw, df_raw.iloc[[3]]], ignore_index=True)
df_raw.loc[[2, 7], "unit_price"] = np.nan

df_raw.to_csv("orders.csv", index=False)
df_raw.to_excel("orders.xlsx", index=False)
df_raw.to_json("orders.json", orient="records", indent=2)

print("Sample files created: orders.csv, orders.xlsx, orders.json")
df_raw.head()


ModuleNotFoundError: No module named 'openpyxl'

## 5. Pandas DataFrame Basics

In [ ]:
df = pd.read_csv("orders.csv")

print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)
print("\nInfo:")
df.info()


In [ ]:
df.describe(include='all')


## 6. Reading CSV / Excel / JSON

Three different readers for three common formats — all produce a DataFrame.

In [6]:
df_from_csv = pd.read_csv("orders.csv")
df_from_excel = pd.read_excel("orders.xlsx")   # requires openpyxl (already installed)
df_from_json = pd.read_json("orders.json")

print("CSV shape:", df_from_csv.shape)
print("Excel shape:", df_from_excel.shape)
print("JSON shape:", df_from_json.shape)

df_from_excel.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'orders.xlsx'

## 7. Filtering

Selecting subsets of rows based on conditions.

In [ ]:
# Single condition
electronics_orders = df[df["category"] == "Electronics"]
print("Electronics orders:", len(electronics_orders))

# Multiple conditions (note the parentheses + & / | operators)
high_value_bangalore = df[(df["city"] == "Bengaluru") & (df["unit_price"] > 500)]
print("High value Bengaluru orders:", len(high_value_bangalore))

# .query() syntax as an alternative
young_customers = df.query("age < 30")
print("Customers under 30:", len(young_customers))

high_value_bangalore


## 8. GroupBy

Split-apply-combine: group rows by a key, then aggregate.

In [ ]:
# Average unit price per category
avg_price_by_category = df.groupby("category")["unit_price"].mean().round(2)
print(avg_price_by_category)

# Multiple aggregations at once
summary = df.groupby("category").agg(
    total_orders=("customer_id", "count"),
    avg_price=("unit_price", "mean"),
    total_quantity=("quantity", "sum")
).round(2)

summary


In [ ]:
# Group by two keys
city_category_summary = df.groupby(["city", "category"])["unit_price"].mean().round(2).unstack()
city_category_summary


## 9. Missing Value Handling

Real-world data almost always has gaps. Key steps: detect, decide (drop vs fill), then fill/drop.

In [ ]:
# Step 1: Detect missing values
print("Missing values per column:\n", df.isnull().sum())
print("\nPercent missing:\n", (df.isnull().mean() * 100).round(1))


In [ ]:
df_clean = df.copy()

# Step 2: Drop rows where a critical field (customer_id) is missing (none here, but good practice)
df_clean = df_clean.dropna(subset=["customer_id"])

# Step 3: Drop exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} duplicate row(s)")

# Step 4: Fill missing numeric values sensibly
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean["quantity"] = df_clean["quantity"].fillna(1)   # assume at least 1 unit
df_clean["unit_price"] = df_clean["unit_price"].fillna(df_clean.groupby("category")["unit_price"].transform("mean"))

# Step 5: Fill missing categorical values with a placeholder
df_clean["city"] = df_clean["city"].fillna("Unknown")

print("\nRemaining missing values:\n", df_clean.isnull().sum())


## 10. Data Preprocessing

Cleaning up formatting/consistency issues (casing, whitespace, types).

In [ ]:
# Standardize text casing
df_clean["name"] = df_clean["name"].str.strip().str.title()
df_clean["city"] = df_clean["city"].str.strip().str.title()
df_clean["category"] = df_clean["category"].str.strip().str.title()

# Correct data types
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])
df_clean["age"] = df_clean["age"].astype(int)
df_clean["quantity"] = df_clean["quantity"].astype(int)

# Ensure numeric columns are non-negative (basic sanity check)
df_clean = df_clean[df_clean["unit_price"] >= 0]

df_clean.dtypes


In [ ]:
df_clean.head(10)


## 11. Feature Engineering Basics

Creating new, more useful columns from existing ones.

In [ ]:
# 1. Derived numeric feature
df_clean["total_amount"] = df_clean["quantity"] * df_clean["unit_price"]

# 2. Date-based features
df_clean["order_month"] = df_clean["order_date"].dt.month_name()
df_clean["order_weekday"] = df_clean["order_date"].dt.day_name()

# 3. Binning a continuous variable into categories
df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=[0, 25, 35, 100],
    labels=["Young Adult", "Adult", "Senior"]
)

# 4. Simple flag/indicator feature
df_clean["is_high_value"] = (df_clean["total_amount"] > df_clean["total_amount"].median()).astype(int)

df_clean[["customer_id", "name", "age", "age_group", "total_amount", "order_month", "order_weekday", "is_high_value"]].head(10)


## 12. Final Check & Save Cleaned Data

In [ ]:
print("Final shape:", df_clean.shape)
print("\nAny missing values left?\n", df_clean.isnull().sum().sum())

df_clean.to_csv("orders_cleaned.csv", index=False)
print("\nSaved cleaned dataset -> orders_cleaned.csv")
df_clean.head()


---
### Summary — what we covered today
1. **NumPy arrays** — creation, indexing, vectorized operations
2. **Broadcasting** — scalar-array and array-array operations of different shapes
3. **Pandas DataFrame** — structure, dtypes, describe/info
4. **Reading CSV/Excel/JSON** — `pd.read_csv`, `pd.read_excel`, `pd.read_json`
5. **Filtering** — boolean masks, multiple conditions, `.query()`
6. **GroupBy** — single/multiple key aggregation, `.agg()`, `.unstack()`
7. **Missing values** — detect, drop, fill (mean/median/group-mean/placeholder)
8. **Preprocessing** — text casing, dtype fixes, sanity checks, deduplication
9. **Feature engineering** — derived columns, date parts, binning, flags

**Next step ideas:** try this same pipeline on a real dataset (e.g. from Kaggle), and explore `sklearn.preprocessing` (StandardScaler, OneHotEncoder) for ML-ready feature engineering.
